# Pruning Whole Model

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google Drive

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = train_images / 255.0
test_images = test_images / 255.0

## CNN Model for MNIST (CNN)

In [5]:
model = keras.Sequential([
    keras.layers.InputLayer(input_shape=(28, 28)),
    keras.layers.Reshape(target_shape=(28, 28, 1)),
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

In [6]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

## Compile & Fit model

In [7]:
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [8]:
hist_base = model.fit(
  train_images,
  train_labels,
  epochs=10,
  validation_split=0.1,
)

Epoch 1/10
1688/1688 [==============================] - 37s 21ms/step - loss: 0.1857 - accuracy: 0.9423 - val_loss: 0.0631 - val_accuracy: 0.9820
Epoch 2/10
1688/1688 [==============================] - 38s 22ms/step - loss: 0.0584 - accuracy: 0.9818 - val_loss: 0.0660 - val_accuracy: 0.9800
Epoch 3/10
1688/1688 [==============================] - 38s 22ms/step - loss: 0.0403 - accuracy: 0.9874 - val_loss: 0.0431 - val_accuracy: 0.9888
Epoch 4/10
1688/1688 [==============================] - 37s 22ms/step - loss: 0.0317 - accuracy: 0.9900 - val_loss: 0.0422 - val_accuracy: 0.9883
Epoch 5/10
1688/1688 [==============================] - 38s 22ms/step - loss: 0.0237 - accuracy: 0.9922 - val_loss: 0.0449 - val_accuracy: 0.9865
Epoch 6/10
1688/1688 [==============================] - 38s 22ms/step - loss: 0.0205 - accuracy: 0.9934 - val_loss: 0.0376 - val_accuracy: 0.9905
Epoch 7/10
1688/1688 [==============================] - 39s 23ms/step - loss: 0.0154 - accuracy: 0.9947 - val_loss: 0.0401 -

In [9]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


In [10]:
model.layers[1].get_weights()

[array([[[[ 1.57714173e-01, -5.83508331e-03, -1.61653664e-02,
            2.49363318e-01,  6.52938113e-02,  1.06829591e-01,
           -1.71385765e-01, -2.76716232e-01, -1.74441591e-01,
            4.45471644e-01,  9.45033357e-02, -2.92737842e-01,
            2.09057033e-01, -2.12433049e-03,  4.11112234e-02,
            1.33311868e-01,  1.77187771e-02,  9.29089114e-02,
            4.53191157e-03, -1.65191703e-02,  3.32915604e-01,
           -4.92629588e-01,  2.01682627e-01, -2.65789870e-02,
           -3.61835420e-01, -5.44804484e-02, -1.44337222e-01,
           -3.30566794e-01, -8.56284052e-03,  2.09492385e-01,
            4.19246167e-01, -3.33595902e-01]],
 
         [[ 1.36669710e-01, -2.66587228e-01,  9.36541408e-02,
            2.13598669e-01, -5.14182774e-03,  7.96611980e-02,
            5.86670032e-03,  4.99681458e-02,  2.09513828e-02,
            2.33950630e-01, -2.24051282e-01,  6.60920888e-02,
            1.57601282e-01, -2.29333758e-01,  1.51049420e-01,
            1.5550623

## Save Baseline Model

In [11]:
model.save('/content/drive/MyDrive/files/save/baseline_model.h5')

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## Fine-tune pre-trained model with pruning (Whole model)
* Scheduler : tfmot.sparsity.keras.PolynomialDecay
    * Initial sparsity : 50%
    * End sparsigy : 80%

In [12]:
batch_size = 128
epochs = 2
validation_split = 0.1

num_images = train_images.shape[0] * (1 - validation_split)
end_step = np.ceil(num_images / batch_size).astype(np.int32) * epochs   #전체 training 마지막 step

pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.50,
                                                               final_sparsity=0.80,
                                                               begin_step=0,
                                                               end_step=end_step)
}

model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)


In [13]:
model_for_pruning.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
model_for_pruning.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_reshap  (None, 28, 28, 1)         1         
 e (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_conv2d  (None, 26, 26, 32)        610       
  (PruneLowMagnitude)                                            
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 32)        1         
 oling2d (PruneLowMagnitude                                      
 )                                                               
                                                                 
 prune_low_magnitude_conv2d  (None, 11, 11, 16)        9234      
 _1 (PruneLowMagnitude)                                          
                                                        

In [14]:
callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep(),
]

hist_pruning = model_for_pruning.fit(train_images, train_labels,
                  batch_size=batch_size, epochs=epochs, validation_split=validation_split,
                  callbacks=callbacks)

_, model_for_pruning_accuracy = model_for_pruning.evaluate(
   test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Pruned test accuracy:', model_for_pruning_accuracy)

Epoch 1/2
422/422 [==============================] - 36s 76ms/step - loss: 0.0196 - accuracy: 0.9941 - val_loss: 0.0418 - val_accuracy: 0.9887
Epoch 2/2
422/422 [==============================] - 30s 71ms/step - loss: 0.0241 - accuracy: 0.9930 - val_loss: 0.0333 - val_accuracy: 0.9902
Baseline test accuracy: 0.9900000095367432
Pruned test accuracy: 0.9890000224113464


In [15]:
model_for_export = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

total, non_zero = 0, 0
for l in model_for_export.layers:
    weights = l.get_weights()
    if len(weights)>0 and type(weights[0]) == np.ndarray:
        size = weights[0].size
        cnt_nonzero = np.count_nonzero(weights[0])
        total += size
        non_zero += cnt_nonzero
        print("[{:<10}] pruning rate : {}".format(l.name, (size - cnt_nonzero) /  size))

print( "Total parameter : {}".format(total))
print( "Non-zero parameter : {}".format(non_zero))
print( "Rate of pruned parmeter : {}".format((total-non_zero)/ total))

[conv2d    ] pruning rate : 0.7986111111111112
[conv2d_1  ] pruning rate : 0.7999131944444444
[dense     ] pruning rate : 0.7999609375
[dense_1   ] pruning rate : 0.8
Total parameter : 57376
Non-zero parameter : 11478
Rate of pruned parmeter : 0.7999511991076408


In [16]:
model_for_export.summary()
model_for_export.layers[1].get_weights()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

[array([[[[ 0.        , -0.        ,  0.        ,  0.        ,
           -0.        , -0.        , -0.        , -0.        ,
           -0.        ,  0.50788486,  0.        , -0.        ,
            0.41204175, -0.        , -0.        , -0.        ,
           -0.        , -0.        ,  0.        ,  0.        ,
            0.3997148 , -0.54865044, -0.        , -0.        ,
           -0.31757936,  0.        ,  0.        ,  0.        ,
            0.        ,  0.        ,  0.43969825, -0.47105253]],
 
         [[ 0.        , -0.        ,  0.        ,  0.        ,
            0.        , -0.        ,  0.        ,  0.        ,
            0.        ,  0.        ,  0.        ,  0.        ,
           -0.        , -0.        ,  0.        , -0.        ,
            0.        , -0.        ,  0.        ,  0.        ,
           -0.        , -0.        , -0.        ,  0.        ,
           -0.3277851 ,  0.        , -0.372677  ,  0.        ,
            0.        ,  0.        ,  0.40264717,  